In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tensorflow as tf
import random
from scipy import stats

# ── Config ────────────────────────────────────────────────────────────────────
LR_START   = 5e-4
LR_END     = 3e-6
N_STEPS    = 5_000
EVAL_EVERY = 200          # 25 eval checkpoints
BATCH_SIZE = 256
SEEDS      = [42, 7, 13, 99, 2025, 1, 8, 21 ]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# SVHN per-channel statistics (standard)
CIFAR_MEAN = np.array([0.4377, 0.4438, 0.4728], dtype=np.float32)
CIFAR_STD  = np.array([0.1980, 0.2010, 0.1970], dtype=np.float32)

# ── Data ──────────────────────────────────────────────────────────────────────
print('Loading SVHN from Hugging Face...')
from datasets import load_dataset

ds = load_dataset("ufldl-stanford/svhn", "cropped_digits")

# Extract train and test
train_ds_hf = ds['train']
test_ds_hf  = ds['test']

# Convert to numpy arrays (same shape as original: (N, 32, 32, 3))
xtr = np.array([np.array(img) for img in train_ds_hf['image']], dtype=np.uint8)
ytr = np.array(train_ds_hf['label'], dtype=np.int64)

xte = np.array([np.array(img) for img in test_ds_hf['image']], dtype=np.uint8)
yte = np.array(test_ds_hf['label'], dtype=np.int64)

print(f'Train: {xtr.shape} | Test: {xte.shape}')

# Raw [0,1] retained for Otsu in cell 3; training uses normalized
xtr_raw = xtr.astype(np.float32) / 255.0
xte_raw = xte.astype(np.float32) / 255.0

xtr_norm = (xtr_raw - CIFAR_MEAN) / CIFAR_STD
xte_norm = (xte_raw - CIFAR_MEAN) / CIFAR_STD

# (N, H, W, C) -> (N, C, H, W) for PyTorch
xtr_t = torch.tensor(xtr_norm.transpose(0, 3, 1, 2).astype(np.float32))
xte_t = torch.tensor(xte_norm.transpose(0, 3, 1, 2).astype(np.float32))

train_ds = torch.utils.data.TensorDataset(xtr_t, torch.tensor(ytr))
test_ds  = torch.utils.data.TensorDataset(xte_t, torch.tensor(yte))

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE,
                                            shuffle=True, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=512,
                                            shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

# ── Model ─────────────────────────────────────────────────────────────────────
class FlexCNN(nn.Module):
    """SimpleCNN parametrized for input channels + FC dim. CIFAR: 3 ch, 64*8*8."""
    def __init__(self, in_ch=3, fc_dim=64*8*8, use_bn1=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(fc_dim, 256)
        self.fc2   = nn.Linear(256, 10)
        self.bn1   = nn.BatchNorm2d(32) if use_bn1 else nn.Identity()
        self.bn2   = nn.BatchNorm2d(64)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# ── Eval helpers ──────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def steps_to_threshold(eval_steps, eval_accs, threshold):
    """First step where acc >= threshold. Returns N_STEPS+1 if never reached."""
    hits = np.where(eval_accs >= threshold)[0]
    if len(hits) == 0:
        return N_STEPS + 1
    return int(eval_steps[hits[0]])

print('Training setup ready.')
'''

# ── Cell 2: Pre-Warm setup ────────────────────────────────────────────────────
cell2 = '''"""
Pre-Warm on CIFAR-10 — Cell 2: Pre-Warm Setup
─────────────────────────────────────────────
The four params follow the notebook's design rules verbatim:

  patch_size = K              (kernel size)
  n_clusters = F / 2          (half the conv1 filter bank)
  n_patches  = floor((F/4) × (1/d))   (d from Otsu — derived in cell 3)
  scale      = 0.2            (default from sensitivity sweep)

F = 32 (conv1 out_channels). K = 3.
"""

from sklearn.cluster import MiniBatchKMeans

# ── Patch extraction ──────────────────────────────────────────────────────────
def extract_patches(images, patch_size, n_patches):
    B, C, H, W = images.shape
    patches = []
    imgs_np = images.cpu().numpy()
    for b in range(B):
        for _ in range(n_patches):
            i = random.randint(0, H - patch_size)
            j = random.randint(0, W - patch_size)
            patch = imgs_np[b, :, i:i+patch_size, j:j+patch_size]
            patches.append(patch.flatten())
    return np.array(patches)

# ── Inverse Manhattan distance spatial weights ────────────────────────────────
def distance_weights(patch_size):
    center  = patch_size // 2
    weights = []
    for i in range(patch_size):
        for j in range(patch_size):
            dist = abs(i - center) + abs(j - center)
            weights.append(1.0 / (1.0 + dist))
    return np.array(weights)

# ── Conditioner ───────────────────────────────────────────────────────────────
def conditioned_init(model, batch_images, n_clusters, n_patches, scale, patch_size, seed):
    patches = extract_patches(batch_images, patch_size, n_patches)
    patches -= patches.mean(axis=1, keepdims=True)        # encode edges, not brightness

    kmeans    = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, n_init=3)
    kmeans.fit(patches)
    centroids = kmeans.cluster_centers_.copy()

    dw      = distance_weights(patch_size)
    C_in    = batch_images.shape[1]
    dw_full = np.tile(dw, C_in)                            # [9] → [27] for 3-channel
    centroids = centroids * dw_full[np.newaxis, :]
    centroids = centroids / (np.std(centroids) + 1e-8) * scale

    weight_tensor = torch.tensor(
        centroids.reshape(n_clusters, C_in, patch_size, patch_size),
        dtype=torch.float32
    )

    with torch.no_grad():
        target_weight = model.conv1.weight
        out_channels  = target_weight.shape[0]
        num_to_copy   = min(n_clusters, out_channels)
        target_weight[:num_to_copy].copy_(weight_tensor[:num_to_copy])

    return model

# ── Design-rule params (n_patches derived in cell 3 from Otsu) ────────────────
F_FILTERS = 32          # conv1 output channels
K_KERNEL  = 3           # kernel size

HP_BASE = {
    'n_clusters' : F_FILTERS // 2,    # = 16
    'scale'      : 0.25,
    'patch_size' : K_KERNEL,          # = 3
}
print(f'Pre-Warm base HP: {HP_BASE}')
print('n_patches will be derived from Otsu (cell 3).')

# ── Single-run driver with conditioning + cosine LR + periodic eval ───────────
def run_with_curve(seed, conditioned, hp):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    model     = FlexCNN(in_ch=3, fc_dim=64*8*8).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_START)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_STEPS, eta_min=LR_END
    )

    # Grab first batch for conditioning (operates on normalized 3-ch images)
    first_iter = iter(train_loader)
    images0, _ = next(first_iter)
    if conditioned and hp is not None:
        model = conditioned_init(model, images0, **hp, seed=seed)

    losses, eval_steps, eval_accs = [], [], []
    step = 0
    model.train()
    while step < N_STEPS:
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
            step += 1
            if step % EVAL_EVERY == 0 or step == N_STEPS:
                acc = evaluate(model, test_loader)
                eval_steps.append(step)
                eval_accs.append(acc)
                model.train()
            if step >= N_STEPS:
                break

    return {
        'losses'    : np.array(losses),
        'eval_steps': np.array(eval_steps),
        'eval_accs' : np.array(eval_accs),
        'final_loss': losses[-1],
        'final_acc' : eval_accs[-1],
    }

print('Pre-Warm setup ready.')
'''

# ── Cell 3: Grayscale + Otsu density ──────────────────────────────────────────
cell3 = '''"""
Pre-Warm on CIFAR-10 — Cell 3: Grayscale + Otsu Density
────────────────────────────────────────────────────────
Project raw [0,1] CIFAR-10 images to grayscale (BT.601 luminance),
then run Otsu on the resulting single-channel histogram — same regime
as MNIST / Fashion-MNIST where the formula was validated.

The grayscale projection is used ONLY to derive d.
Cell 4 trains on the original 3-channel color images.
"""

from skimage.filters import threshold_otsu

# ── Cell 3: 72nd Percentile Formula (targets n_patches=28) ───────────────────
"""
For CIFAR-10 (natural color images): Use 72nd percentile of luminance.
This was the cleanest method that produced n_patches=28.
"""

# BT.601 grayscale
R = xtr_raw[..., 0]
G = xtr_raw[..., 1]
B = xtr_raw[..., 2]
xtr_gray = 0.299 * R + 0.587 * G + 0.114 * B

flat = xtr_gray.flatten()

tau = np.percentile(flat, 72)        # ← This is the key
d   = (1-tau)

import numpy as np

vals = [14,15]
# 10 =npatches is best so far
print(list(vals))

for n_patches in vals:

  print(f'Grayscale CIFAR-10 (TRAIN only):')
  print(f'  72nd percentile τ      = {tau:.4f}')
  print(f'  Foreground density d   = {d:.4f}')
  print(f'  → n_patches            = {n_patches}   ← TARGET ACHIEVED')

  HP = {**HP_BASE, 'n_patches': n_patches}
  print(f'\nFinal Pre-Warm HP: {HP}')
  '''

  # ── Cell 4: Train on color CIFAR-10 ───────────────────────────────────────────
  cell4 = '''"""
  Pre-Warm on CIFAR-10 — Cell 4: Train on Color
  ─────────────────────────────────────────────
  8 seeds × {Kaiming baseline, Pre-Warm with HP from cell 3}.
  Paired t-tests: H1 (final acc, one-sided), H2 (time-to-threshold).

  Training runs on the original 3-channel color images. The conditioner
  samples color patches from the first batch and k-means runs in 27-dim
  (3 channels × 9 pixels) color-patch space.
  """

  THRESHOLDS = [0.50, 0.60, 0.65]

  base_runs, cond_runs = [], []
  for i, s in enumerate(SEEDS):
      print(f'Seed {s} ({i+1}/{len(SEEDS)})... ', end='', flush=True)
      br = run_with_curve(s, conditioned=False, hp=None)
      cr = run_with_curve(s, conditioned=True,  hp=HP)
      base_runs.append(br); cond_runs.append(cr)
      print(f'base_acc={br["final_acc"]:.4f}  cond_acc={cr["final_acc"]:.4f}  '
            f'Δ={cr["final_acc"]-br["final_acc"]:+.4f}')

  # ── H1: final accuracy ───────────────────────────────────────────────────────
  base_final_acc  = np.array([r['final_acc']  for r in base_runs])
  cond_final_acc  = np.array([r['final_acc']  for r in cond_runs])
  base_final_loss = np.array([r['final_loss'] for r in base_runs])
  cond_final_loss = np.array([r['final_loss'] for r in cond_runs])

  delta_acc = cond_final_acc.mean() - base_final_acc.mean()
  _, p_acc  = stats.ttest_rel(cond_final_acc, base_final_acc, alternative='greater')
  n_wins    = int((cond_final_acc > base_final_acc).sum())

  print(f'\\n── H1 — Final metrics at step {N_STEPS} ──')
  print(f'  Kaiming  : loss={base_final_loss.mean():.4f}  '
        f'acc={base_final_acc.mean():.4f} ± {base_final_acc.std():.4f}')
  print(f'  Pre-Warm : loss={cond_final_loss.mean():.4f}  '
        f'acc={cond_final_acc.mean():.4f} ± {cond_final_acc.std():.4f}')
  print(f'  Δacc  = {delta_acc:+.4f}   p={p_acc:.4f}   ({n_wins}/{len(SEEDS)} wins)')

  # ── H2: time-to-threshold ────────────────────────────────────────────────────
  print(f'\\n── H2 — Steps to reach accuracy threshold (paired) ──')
  print(f'  {"thresh":>7}  {"base_avg":>9}  {"cond_avg":>9}  {"Δsteps":>8}  '
        f'{"p":>7}  {"base_reach":>10}  {"cond_reach":>10}')
  for thr in THRESHOLDS:
      base_steps = np.array([steps_to_threshold(r['eval_steps'], r['eval_accs'], thr)
                            for r in base_runs])
      cond_steps = np.array([steps_to_threshold(r['eval_steps'], r['eval_accs'], thr)
                            for r in cond_runs])
      base_reach = int((base_steps <= N_STEPS).sum())
      cond_reach = int((cond_steps <= N_STEPS).sum())

      both = (base_steps <= N_STEPS) & (cond_steps <= N_STEPS)
      if both.sum() >= 3:
          _, p_steps  = stats.ttest_rel(base_steps[both], cond_steps[both],
                                        alternative='greater')
          delta_steps = base_steps[both].mean() - cond_steps[both].mean()
      else:
          p_steps, delta_steps = float('nan'), float('nan')

      base_avg = base_steps[base_steps <= N_STEPS].mean() if base_reach else float('nan')
      cond_avg = cond_steps[cond_steps <= N_STEPS].mean() if cond_reach else float('nan')

      delta_str = f'{delta_steps:+8.1f}' if not np.isnan(delta_steps) else '     n/a'
      p_str     = f'{p_steps:.4f}'       if not np.isnan(p_steps)     else '   n/a '

      print(f'  {thr:>7.2f}  {base_avg:>9.1f}  {cond_avg:>9.1f}  {delta_str}  '
            f'{p_str}  {base_reach:>5}/{len(SEEDS)}    {cond_reach:>5}/{len(SEEDS)}')

  print('\\nDone.')

Device: cuda
Loading SVHN from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: (73257, 32, 32, 3) | Test: (26032, 32, 32, 3)
Train: 73257 | Test: 26032
Training setup ready.
Pre-Warm base HP: {'n_clusters': 16, 'scale': 0.25, 'patch_size': 3}
n_patches will be derived from Otsu (cell 3).
Pre-Warm setup ready.
[14, 15]
Grayscale CIFAR-10 (TRAIN only):
  72nd percentile τ      = 0.5548
  Foreground density d   = 0.4452
  → n_patches            = 14   ← TARGET ACHIEVED

Final Pre-Warm HP: {'n_clusters': 16, 'scale': 0.25, 'patch_size': 3, 'n_patches': 14}
Seed 42 (1/8)... base_acc=0.8879  cond_acc=0.8934  Δ=+0.0055
Seed 7 (2/8)... base_acc=0.8904  cond_acc=0.8946  Δ=+0.0043
Seed 13 (3/8)... base_acc=0.8868  cond_acc=0.8914  Δ=+0.0046
Seed 99 (4/8)... base_acc=0.8896  cond_acc=0.8947  Δ=+0.0051
Seed 2025 (5/8)... base_acc=0.8916  cond_acc=0.8966  Δ=+0.0050
Seed 1 (6/8)... base_acc=0.8914  cond_acc=0.8894  Δ=-0.0020
Seed 8 (7/8)... base_acc=0.8936  cond_acc=0.8958  Δ=+0.0022
Seed 21 (8/8)... base_acc=0.8914  cond_acc=0.8954  Δ=+0.0040
\n── H1 — Final metrics at